# HR Attrition Risk Prediction: End-to-End Workflow

Run cells in order. Training requires the MLflow server configured by `MLFLOW_TRACKING_URI`.


In [1]:
from pathlib import Path
import os, sys, joblib, pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from app.config import settings
from app.data import (sync_data_from_s3, validate_source_data, preprocess_data, feature_engineering, feature_selection, exploratory_data_analysis)
from app.train import split_data, handle_imbalance_data, train_and_track_models
from app.evaluate import evaluate_classifier, evaluate_risk_bands
from app.features import FINAL_FEATURES, TARGET_COLUMN
from app.inference import get_inference_services
from app.explain import generate_llm_explanation
from app.schemas import AttritionInput, ExplainInput

print(f'Project root: {ROOT}')
print(f'Data source: {settings.DATA_SOURCE}')
print(f'MLflow URI: {settings.MLFLOW_TRACKING_URI}')

Project root: D:\Cybersoft__Course\MLOps_01\buoi10\Đề bài\04\hr_attrition_risk_prediction
Data source: s3
MLflow URI: http://localhost:5001


## 1. Ingest and validate source data

`sync_data_from_s3` skips safely for local data. `validate_source_data` performs the temporal joins and writes `data/merged/merged_data.parquet`.


In [2]:
ingestion_report = sync_data_from_s3()
validation_report = validate_source_data()
display(pd.DataFrame([ingestion_report, validation_report]))

merged_df = pd.read_parquet(settings.MERGED_DATA_PATH)
display(merged_df.head())
print(f'Merged dataset shape: {merged_df.shape}')

,status,bucket,prefix,files,data_dir,validated_at,rows,columns,source_file
0,downloaded,baitap-buoi9-bucket,data/,"[attrition_labels.csv, employees.csv, engageme...",data\raw,NaN,NaN,NaN,NaN
1,passed,NaN,NaN,NaN,NaN,2026-10-10T16:28:27.830633+00:00,128540.0,40.0,6.0


,employee_id,snapshot_date,department,job_level,label,role,location,employment_type,hire_date,manager_id,...,peer_feedback_score,review_sentiment_score,week_start,tasks_completed,late_tasks,overtime_hours,meeting_hours,focus_hours,after_hours_messages,absent_days
0,E000001,2025-01-31,Customer Success,3,0,Senior CSM,Ho Chi Minh City,Full-time,2016-01-06,M00435,...,4.27,0.876,2025-01-27,61,7,9.5,11.9,28.4,17,0
1,E000002,2025-01-31,Operations,3,0,Senior Operations Specialist,Ho Chi Minh City,Full-time,2023-11-13,M00204,...,3.11,0.633,2025-01-27,47,1,8.5,10.6,26.5,8,0
2,E000003,2025-01-31,Marketing,2,0,Marketing Specialist,Ho Chi Minh City,Contract,2017-02-09,M00367,...,3.46,0.965,2025-01-27,27,2,8.7,7.9,23.0,9,0
3,E000004,2025-01-31,Customer Success,1,0,CS Associate,Da Nang,Full-time,2022-12-19,M00795,...,3.75,0.852,2025-01-27,52,4,10.9,1.3,27.9,12,0
4,E000005,2025-01-31,Sales,1,0,Sales Associate,Ha Noi,Full-time,2016-02-29,M00460,...,3.04,0.701,2025-01-27,51,5,10.4,4.7,31.0,4,0


Merged dataset shape: (128540, 40)


## 2. Explore, preprocess, and select features

EDA files are written to `artifacts/eda_charts`; feature selection writes the model contract to `data/processed/feature_selected.parquet`.


In [3]:
eda_report = exploratory_data_analysis()
preprocessing_report = preprocess_data()
engineering_report = feature_engineering()
selection_report = feature_selection()
display(pd.DataFrame([eda_report, preprocessing_report, engineering_report, selection_report]))

feature_df = pd.read_parquet(settings.FEATURE_SELECTED_PATH)
display(feature_df.head())
print(f'Selected feature dataset shape: {feature_df.shape}')

,status,eda_dir,charts,sample_count,rows,columns,target_column,output,selected_features,num_features
0,completed,artifacts\eda_charts,[artifacts\eda_charts\attrition_by_department....,128540.0,NaN,NaN,NaN,NaN,NaN,NaN
1,completed,NaN,NaN,NaN,128540.0,"[employee_id, snapshot_date, department, job_l...",label,D:\Cybersoft__Course\MLOps_01\buoi10\Đề bài\04...,NaN,NaN
2,completed,NaN,NaN,NaN,128540.0,"[employee_id, snapshot_date, department, job_l...",NaN,D:\Cybersoft__Course\MLOps_01\buoi10\Đề bài\04...,NaN,NaN
3,completed,NaN,NaN,NaN,128540.0,NaN,NaN,NaN,"[employee_id, tasks_completed, late_tasks, ove...",34.0


,employee_id,tasks_completed,late_tasks,overtime_hours,meeting_hours,focus_hours,after_hours_messages,absent_days,performance_score,goal_completion_rate,...,tenure_months,days_since_last_promotion,leave_type,is_unplanned,approved_by_manager,department,role,location,employment_type,label
0,E000001,61,7,9.5,11.9,28.4,17,0,3.65,95.5,...,110.433333,3174,Sick,False,True,Customer Success,Senior CSM,Ho Chi Minh City,Full-time,0
1,E000002,47,1,8.5,10.6,26.5,8,0,3.27,92.5,...,14.833333,445,Sick,False,True,Operations,Senior Operations Specialist,Ho Chi Minh City,Full-time,0
2,E000003,27,2,8.7,7.9,23.0,9,0,3.50,79.4,...,97.100000,2697,Personal,False,True,Marketing,Marketing Specialist,Ho Chi Minh City,Contract,0
3,E000004,52,4,10.9,1.3,27.9,12,0,3.48,95.8,...,25.800000,774,Annual,True,True,Customer Success,CS Associate,Da Nang,Full-time,0
4,E000005,51,5,10.4,4.7,31.0,4,0,3.06,96.4,...,108.633333,2962,Unpaid,True,True,Sales,Sales Associate,Ha Noi,Full-time,0


Selected feature dataset shape: (128540, 34)


## 3. Split, balance, and train models

`train_and_track_models` logs nested MLflow runs, selects the highest-ROC-AUC candidate, and writes the model and serving metadata.


In [5]:
split_report = split_data()
balance_report = handle_imbalance_data()
display(pd.DataFrame([split_report, balance_report]))

train_df = pd.read_parquet(settings.TRAIN_BALANCED_PATH)
test_df = pd.read_parquet(settings.TEST_BALANCED_PATH)
training_summary = train_and_track_models(train_df, test_df)
metrics_df = pd.DataFrame(training_summary['full_metrics']).sort_values('roc_auc', ascending=False)
display(metrics_df)
print(f"Best model: {training_summary['best_model']['Model']}")

,train_rows,test_rows,random_state,status,train_original_rows,train_balanced_rows,train_label_distribution,test_original_rows,test_balanced_rows,test_label_distribution,output
0,102832.0,25708.0,42.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,completed,102832.0,49421.0,"{'train': {0: 98423, 1: 4409}, 'train_balanced...",25708.0,3402.0,"{'test': {0: 24606, 1: 1102}, 'test_balanced':...",[D:\Cybersoft__Course\MLOps_01\buoi10\Đề bài\0...


2026/10/11 00:20:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/11 00:20:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/11 00:20:59 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\LOQ\AppData\Local\Temp\tmpuucqydqm\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.9.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


🏃 View run DecisionTree at: http://localhost:5001/#/experiments/1/runs/e76a8a111436489e90b2ab12d730d1bf
🧪 View experiment at: http://localhost:5001/#/experiments/1


2026/10/11 00:21:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/11 00:21:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest at: http://localhost:5001/#/experiments/1/runs/fb9096c624e44832beb00f8193e64aea
🧪 View experiment at: http://localhost:5001/#/experiments/1


...

🏃 View run XGBoost at: http://localhost:5001/#/experiments/1/runs/bf9377040a5c4894980428c5aca05238
🧪 View experiment at: http://localhost:5001/#/experiments/1


2026/10/11 00:24:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/11 00:24:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: http://localhost:5001/#/experiments/1/runs/87f7efa247bf4854940767ce89d1a85b
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run model_training_batch at: http://localhost:5001/#/experiments/1/runs/1b6812b1355f4d74aa161df2ecd7e1f8
🧪 View experiment at: http://localhost:5001/#/experiments/1


,Model,accuracy,precision,recall,f1,roc_auc,pr_auc,risk_distribution
0,XGBoost,0.888301,0.918794,0.718693,0.806517,0.952903,0.918479,"{'CRITICAL': {'total_employees': 743, 'actual_..."
1,LightGBM,0.894474,0.884178,0.775862,0.826486,0.952344,0.918924,"{'CRITICAL': {'total_employees': 769, 'actual_..."
2,GradientBoosting,0.894474,0.882595,0.777677,0.826821,0.948433,0.912432,"{'CRITICAL': {'total_employees': 732, 'actual_..."
3,RandomForest,0.827160,0.947735,0.493648,0.649165,0.940225,0.893327,"{'CRITICAL': {'total_employees': 144, 'actual_..."
4,DecisionTree,0.818048,0.691211,0.792196,0.738266,0.872000,0.758482,"{'CRITICAL': {'total_employees': 857, 'actual_..."


Best model: XGBoost


## 4. Evaluate the persisted best model


In [6]:
pipeline = joblib.load(settings.BEST_MODEL_PATH)
X_test, y_test = test_df[FINAL_FEATURES], test_df[TARGET_COLUMN]
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)
metrics = evaluate_classifier(y_test, y_pred, y_proba)
risk_band_report = evaluate_risk_bands(y_test, y_proba)
display(pd.DataFrame([metrics]))
display(pd.DataFrame(risk_band_report['bands_summary']).T)

,accuracy,precision,recall,f1,roc_auc,pr_auc
0,0.888301,0.918794,0.718693,0.806517,0.952903,0.918479


,total_employees,actual_attrition_count,actual_attrition_rate
CRITICAL,743.0,705.0,0.9489
HIGH,119.0,87.0,0.7311
MEDIUM,101.0,66.0,0.6535
LOW,2439.0,244.0,0.1000


## 5. Score an employee and generate a guardrailed explanation

The explanation service uses the configured LLM when available, otherwise its deterministic safe fallback.


In [7]:
sample_row = test_df.dropna(subset=FINAL_FEATURES).iloc[0]
sample_payload = sample_row[FINAL_FEATURES].to_dict()
attrition_input = AttritionInput(**sample_payload)
prediction = get_inference_services().predict_one(attrition_input)
display(pd.DataFrame([prediction.model_dump(exclude={'top_drivers'})]))
display(pd.DataFrame([driver.model_dump() for driver in prediction.top_drivers]))

explain_input = ExplainInput(
    churn_probability=prediction.churn_probability,
    risk_band=prediction.risk_band,
    top_drivers=[driver.model_dump() for driver in prediction.top_drivers],
    department=sample_payload['department'],
    role=sample_payload['role'],
)
explanation = generate_llm_explanation(explain_input)
display(explanation)

,churn_probability,risk_band,recommended_action,model_name,model_version
0,0.0001,LOW,Duy trì lộ trình đào tạo và môi trường làm việ...,Pipeline,unknown


,feature,value,impact
0,days_since_last_promotion,2093.0,Thời gian chưa được thăng chức dài (2093 ngày)
1,unplanned_leave,3.0,Có đợt nghỉ đột xuất không lên lịch trước


{'explanation_text': 'Nhân viên được ghi nhận ở nhóm rủi ro LOW với xác suất dự báo 0.0%. Các tín hiệu chính từ dữ liệu quan sát gồm: Thời gian chưa được thăng chức dài (2093 ngày); Có đợt nghỉ đột xuất không lên lịch trước. Đây là chỉ số cảnh báo sớm để hỗ trợ người quản lý chủ động đồng hành và hỗ trợ, không phải là kết luận chắc chắn về quyết định cá nhân của nhân viên.',
 'suggested_topics': ['Rà soát khối lượng công việc và cân bằng thời gian (Work-Life Balance)',
  'Định hướng mục tiêu phát triển và kế hoạch thăng chức dài hạn',
  'Mức độ hỗ trợ và phản hồi định kỳ từ người quản lý trực tiếp'],
 'guardrail_status': 'passed'}